# REVELA Spatial Clustering (DBSCAN): Cluster-Quality Evaluation

This Google Colab notebook will evaluate the internal cluster quality of REVELA's **Nearby flagged records** DBSCAN. It will calculate the Silhouette Score using Haversine distances, compare the current operational settings (eps = 20 m; MinPts = 3) with a small offline sensitivity grid, and report cluster count, noise count, coverage, and cluster sizes.

This is an offline evaluation only. It will not change production settings, scores, filters, or source records. The Silhouette Score will not be interpreted as detection accuracy or proof of a violation.

## Data needed before running

Upload a CSV named `dbscan_eligible_records.csv`, with **one row per geospatial log record** included in the DBSCAN input for the time period and filters being evaluated. Required columns:

- `logID`: unique geospatial-log identifier
- `flagColor`: effective flag color at the time of the evaluation; the file must include eligible Red **and** Black records
- `latitude`, `longitude`: valid decimal coordinates

Recommended additional column: `barangayName` for describing how clusters cross barangay boundaries. Do not include business names, addresses, or personal information; they are not needed for this calculation.

The export will need to use the same eligible-record definition as the backend: effective Red/Black; non-null coordinates; and at least one of place ID, reporter ID, stored Orange status, or linked inspection report. It will also need the same barangay, effective-color, and detection-date selections that you intend to evaluate. Registry/business and inspection filters will not be applied by the current analytics chart.

**Important:** your earlier `Detection_runs.csv` summary showed 216 Red-only rows. That file will not exactly reproduce the production DBSCAN input because the backend includes Black records too, and the export may not include the full eligibility/filter conditions. Do not describe a Red-only exploratory run as a production reproduction.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from google.colab import files
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import haversine_distances

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 100)
print("Required Colab analysis libraries imported successfully.")

Colab will import the analysis libraries above without installing anything. Upload `dbscan_eligible_records.csv` in the next cell. If that file has not yet been exported, stop here and obtain an eligible Red/Black geospatial-record export using the definition above; the existing Red-only detection export is not a substitute.

In [ ]:
INPUT_FILENAME = "dbscan_eligible_records.csv"
COLAB_DIR = Path("/content")
input_path = COLAB_DIR / INPUT_FILENAME

if not input_path.is_file():
    print(f"Upload {INPUT_FILENAME} containing the eligible Red/Black DBSCAN input records.")
    uploaded = files.upload()
    for filename, content in uploaded.items():
        (COLAB_DIR / filename).write_bytes(content)
    input_path = COLAB_DIR / INPUT_FILENAME

if not input_path.is_file():
    raise FileNotFoundError(
        f"Required input {INPUT_FILENAME!r} was not uploaded. "
        "Do not substitute the Red-only Detection_runs.csv if the goal is to reproduce production DBSCAN."
    )
print(f"Using input file: {input_path}")

The next cell will inspect the uploaded schema and values, report missing coordinates and duplicate log IDs, and confirm which flag colors are present. It will exclude records lacking coordinates because the production DBSCAN query also requires coordinates. Duplicate or invalid identifiers/coordinates will stop the evaluation so they can be corrected at the source rather than silently altering the input.

In [ ]:
raw_df = pd.read_csv(input_path, dtype={"logID": "string", "flagColor": "string"}, keep_default_na=False)
required_columns = {"logID", "flagColor", "latitude", "longitude"}
missing_columns = sorted(required_columns.difference(raw_df.columns))
if missing_columns:
    raise ValueError(f"Input CSV is missing required columns: {missing_columns}")
if raw_df.empty:
    raise ValueError("The DBSCAN input CSV has no records.")

raw_df["logID"] = raw_df["logID"].astype("string").str.strip()
raw_df["flagColor"] = raw_df["flagColor"].astype("string").str.strip()
raw_df["latitude"] = pd.to_numeric(raw_df["latitude"], errors="coerce")
raw_df["longitude"] = pd.to_numeric(raw_df["longitude"], errors="coerce")

missing_log_ids = int(raw_df["logID"].eq("").sum())
duplicate_log_rows = int(raw_df["logID"].ne("").sum() - raw_df.loc[raw_df["logID"].ne(""), "logID"].nunique())
missing_coordinate_rows = int((raw_df["latitude"].isna() | raw_df["longitude"].isna()).sum())
print(f"Input rows: {len(raw_df):,}")
print(f"Rows with missing logID: {missing_log_ids:,}")
print(f"Repeated logID rows: {duplicate_log_rows:,}")
print(f"Rows with missing/non-numeric coordinates: {missing_coordinate_rows:,}")
print("Flag colors in supplied file:", raw_df["flagColor"].value_counts(dropna=False).to_dict())

if missing_log_ids or duplicate_log_rows:
    raise ValueError("Each source geospatial record must have one unique non-empty logID. Resolve duplicates/missing IDs in the export first.")

eligible_df = raw_df.loc[raw_df["flagColor"].str.casefold().isin(["red", "black"])].copy()
eligible_df = eligible_df.dropna(subset=["latitude", "longitude"]).copy()
if eligible_df.empty:
    raise ValueError("No Red/Black records with valid coordinates remain after required eligibility checks.")
if not eligible_df["latitude"].between(-90, 90).all() or not eligible_df["longitude"].between(-180, 180).all():
    raise ValueError("Latitude/longitude values must be valid decimal degrees (latitude -90..90, longitude -180..180).")

available_colors = set(eligible_df["flagColor"].str.casefold())
if available_colors != {"red", "black"}:
    print("WARNING: Both Red and Black are not present. Results will describe only the supplied colors and will not reproduce the production Red/Black input.")
print(f"Records retained for clustering: {len(eligible_df):,}")
print("Retained flag colors:", eligible_df["flagColor"].value_counts().to_dict())
if "barangayName" in eligible_df.columns:
    print(f"Barangays represented: {eligible_df['barangayName'].nunique(dropna=True):,}")
display(eligible_df[[column for column in ["logID", "flagColor", "latitude", "longitude", "barangayName"] if column in eligible_df.columns]].head(5))

This check establishes the analysis unit: one eligible geospatial log record, not one distinct business. If the file is missing Black records or was not exported using the production eligibility rules and intended filters, the calculations below will still describe the uploaded subset, but they will not validate the production chart. No records will be inferred from missing data.

In [ ]:
# These are the current backend settings; alternatives are sensitivity checks only.
BASELINE_EPS_METERS = 20.0
BASELINE_MIN_SAMPLES = 3
EPS_VALUES_METERS = [10.0, 15.0, 20.0, 25.0, 30.0, 40.0]
MIN_SAMPLES_VALUES = [2, 3, 4, 5]
EARTH_RADIUS_METERS = 6_371_008.8

coords_degrees = eligible_df[["latitude", "longitude"]].to_numpy(dtype=float)
coords_radians = np.radians(coords_degrees)
distance_matrix_radians = haversine_distances(coords_radians)

def evaluate_dbscan(eps_meters, min_samples):
    eps_radians = eps_meters / EARTH_RADIUS_METERS
    labels = DBSCAN(
        eps=eps_radians,
        min_samples=min_samples,
        algorithm="ball_tree",
        metric="haversine",
    ).fit_predict(coords_radians)
    clustered_mask = labels != -1
    clustered_labels = labels[clustered_mask]
    cluster_sizes = pd.Series(clustered_labels).value_counts().sort_values(ascending=False)
    cluster_count = int(cluster_sizes.size)
    clustered_count = int(clustered_mask.sum())
    noise_count = int((~clustered_mask).sum())
    score = None
    score_note = "Silhouette unavailable: DBSCAN produced fewer than two non-noise clusters."

    if cluster_count >= 2 and clustered_count > cluster_count:
        clustered_distances = distance_matrix_radians[np.ix_(clustered_mask, clustered_mask)]
        score = float(silhouette_score(clustered_distances, clustered_labels, metric="precomputed"))
        score_note = "Silhouette computed on clustered records only; noise was excluded."

    result = {
        "eps_m": float(eps_meters),
        "min_samples": int(min_samples),
        "cluster_count": cluster_count,
        "clustered_records": clustered_count,
        "noise_records": noise_count,
        "clustered_percent": 100.0 * clustered_count / len(labels),
        "noise_percent": 100.0 * noise_count / len(labels),
        "silhouette": score,
        "silhouette_records": clustered_count if score is not None else 0,
        "largest_cluster_size": int(cluster_sizes.iloc[0]) if cluster_count else 0,
        "cluster_sizes": ", ".join(str(int(size)) for size in cluster_sizes.tolist()),
        "interpretation_note": score_note,
    }
    return result, labels

baseline_result, baseline_labels = evaluate_dbscan(BASELINE_EPS_METERS, BASELINE_MIN_SAMPLES)
print("Current-settings baseline (20 m, MinPts 3):")
display(pd.DataFrame([baseline_result]))

The baseline will reproduce the current spatial distance setup: coordinates will be converted from decimal degrees to radians, Haversine distance will be used, and 20 m will be converted to angular distance using the backend's Earth-radius constant (6,371,008.8 m). MinPts = 3 will include the point itself, as in scikit-learn DBSCAN. Silhouette will be computed only across non-noise records and only when at least two clusters exist. Therefore, every score must be read together with clustered coverage, noise share, and cluster sizes.

In [ ]:
sensitivity_rows = []
sensitivity_labels = {}
for min_samples in MIN_SAMPLES_VALUES:
    for eps_meters in EPS_VALUES_METERS:
        result, labels = evaluate_dbscan(eps_meters, min_samples)
        result["is_current_baseline"] = (
            eps_meters == BASELINE_EPS_METERS and min_samples == BASELINE_MIN_SAMPLES
        )
        sensitivity_rows.append(result)
        sensitivity_labels[(eps_meters, min_samples)] = labels

sensitivity_df = pd.DataFrame(sensitivity_rows)
display(sensitivity_df[[
    "eps_m", "min_samples", "cluster_count", "clustered_records", "noise_records",
    "clustered_percent", "silhouette", "silhouette_records", "largest_cluster_size", "cluster_sizes",
    "is_current_baseline",
]].sort_values(["min_samples", "eps_m"]).reset_index(drop=True))

valid_scores = sensitivity_df.dropna(subset=["silhouette"])
if valid_scores.empty:
    print("No tested parameter combination produced at least two non-noise clusters; Silhouette cannot rank these settings.")
else:
    best_row = valid_scores.loc[valid_scores["silhouette"].idxmax()]
    print(
        "Highest observed Silhouette among tested settings (not an automatic recommendation): "
        f"eps={best_row['eps_m']:.0f} m, MinPts={int(best_row['min_samples'])}, "
        f"score={best_row['silhouette']:.3f}, coverage={best_row['clustered_percent']:.1f}%, "
        f"clusters={int(best_row['cluster_count'])}, sizes={best_row['cluster_sizes']}"
    )
baseline_score = baseline_result["silhouette"]
if baseline_score is None:
    print("Current production baseline has no valid Silhouette Score for this data snapshot.")
else:
    print(f"Current 20 m / MinPts 3 Silhouette: {baseline_score:.3f}; clustered coverage: {baseline_result['clustered_percent']:.1f}%.")

The parameter grid will provide an offline sensitivity comparison, not a statistically trained model or an automatic recalibration. A higher Silhouette Score alone will not determine a new operational setting: the comparison will also consider how many records are noise, how much data are clustered, the number and sizes of clusters, and whether groups remain interpretable for field operations. There is no universal score cutoff that by itself proves over-clustering or under-clustering.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for min_samples in MIN_SAMPLES_VALUES:
    subset = sensitivity_df.loc[sensitivity_df["min_samples"].eq(min_samples)].sort_values("eps_m")
    axes[0].plot(
        subset["eps_m"],
        subset["silhouette"],
        marker="o",
        label=f"MinPts = {min_samples}",
    )
axes[0].axvline(BASELINE_EPS_METERS, color="#555555", linestyle="--", linewidth=1, label="Current eps = 20 m")
axes[0].set_title("Silhouette by eps and MinPts")
axes[0].set_xlabel("Neighborhood radius, eps (meters)")
axes[0].set_ylabel("Silhouette (clustered records only)")
axes[0].set_ylim(-1, 1)
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.2)

for min_samples in MIN_SAMPLES_VALUES:
    subset = sensitivity_df.loc[sensitivity_df["min_samples"].eq(min_samples)].sort_values("eps_m")
    axes[1].plot(
        subset["eps_m"],
        subset["clustered_percent"],
        marker="o",
        label=f"MinPts = {min_samples}",
    )
axes[1].axvline(BASELINE_EPS_METERS, color="#555555", linestyle="--", linewidth=1, label="Current eps = 20 m")
axes[1].set_title("Clustered coverage by eps and MinPts")
axes[1].set_xlabel("Neighborhood radius, eps (meters)")
axes[1].set_ylabel("Records assigned to clusters (%)")
axes[1].set_ylim(0, 100)
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.2)

fig.suptitle("Offline DBSCAN sensitivity analysis")
fig.tight_layout()
plt.show()

The left plot will show separation/cohesion only for records assigned to non-noise clusters; the right plot will show the fraction of all eligible records included in clusters. If a parameter setting appears to improve Silhouette while sharply reducing coverage or fragmenting records into tiny groups, that trade-off will need to be stated rather than hidden.

In [ ]:
baseline_plot = eligible_df.copy()
baseline_plot["cluster"] = baseline_labels
baseline_plot["cluster_label"] = baseline_plot["cluster"].map(
    lambda label: "Noise" if label == -1 else f"Cluster {label}"
)
cluster_sizes = baseline_plot.loc[baseline_plot["cluster"].ne(-1), "cluster"].value_counts()
largest_clusters = set(cluster_sizes.head(8).index.tolist())
baseline_plot["plot_group"] = baseline_plot["cluster"].map(
    lambda label: "Noise" if label == -1 else (
        f"Cluster {label}" if label in largest_clusters else "Other clusters"
    )
)

fig, ax = plt.subplots(figsize=(9, 7))
category_order = ["Noise"] + [f"Cluster {label}" for label in cluster_sizes.head(8).index.tolist()]
if (baseline_plot["plot_group"] == "Other clusters").any():
    category_order.append("Other clusters")
colors = plt.cm.tab10(np.linspace(0, 1, max(len(category_order), 1)))
for color, category in zip(colors, category_order):
    subset = baseline_plot.loc[baseline_plot["plot_group"].eq(category)]
    if not subset.empty:
        ax.scatter(subset["longitude"], subset["latitude"], s=26, alpha=0.8, label=category, color=color)
ax.set_title("Current baseline clusters (20 m, MinPts 3)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend(title="DBSCAN label", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

if "barangayName" in baseline_plot.columns:
    grouped = baseline_plot.loc[baseline_plot["cluster"].ne(-1)].groupby("cluster")["barangayName"].nunique()
    cross_barangay = int(grouped.gt(1).sum())
    print(f"Non-noise clusters spanning more than one barangay: {cross_barangay} of {len(grouped)}")
else:
    print("No barangayName column was supplied; cross-barangay cluster counts cannot be summarized.")

The baseline map will use latitude/longitude coordinates and distinguish noise from the largest clusters. It will not treat barangay boundaries as clustering barriers. If the export includes barangay names, the accompanying count will identify how many DBSCAN clusters contain records from multiple barangays.

In [ ]:
baseline_label = (
    f"{baseline_result['silhouette']:.3f}"
    if baseline_result["silhouette"] is not None
    else "unavailable (requires at least two non-noise clusters)"
)
print("Evaluation snapshot summary")
print(f"Eligible geospatial records analyzed: {len(eligible_df):,}")
print(f"Current baseline: eps={BASELINE_EPS_METERS:.0f} m, MinPts={BASELINE_MIN_SAMPLES}")
print(f"Baseline clusters: {baseline_result['cluster_count']}; cluster sizes: {baseline_result['cluster_sizes'] or 'none'}")
print(f"Baseline noise: {baseline_result['noise_records']:,} ({baseline_result['noise_percent']:.1f}%)")
print(f"Baseline clustered coverage: {baseline_result['clustered_records']:,} ({baseline_result['clustered_percent']:.1f}%)")
print(f"Baseline Silhouette: {baseline_label}")
print("Parameter changes were evaluated offline only; production settings were not modified.")

## Manuscript wording (future tense)

> **Spatial Clustering (DBSCAN)—Cluster-Quality Assessment.** The DBSCAN procedure used to group nearby flagged geospatial records will be evaluated using the Silhouette Score, calculated with the Haversine distance metric. The current operational settings will be treated as the baseline (eps = 20 m; MinPts = 3, including the point itself). The evaluation will report the number and sizes of non-noise clusters, the number and proportion of noise records, and the proportion of eligible records assigned to clusters. A Silhouette Score will be reported only when at least two non-noise clusters are present; noise records will be excluded from that score and reported separately. Alternative eps and MinPts settings will be compared offline on the same eligible records. Any candidate setting will be discussed in terms of Silhouette, coverage, noise, cluster sizes, and operational interpretability; the score alone will not trigger automatic recalibration. This internal cluster-quality assessment will not be interpreted as detection accuracy or evidence of a violation.

After running the notebook, replace the current-setting and final-parameter statements with measured values from the verified output. Do not claim that parameters were recalibrated unless a separate decision is made to change production settings.

### Q&A

- **What will this evaluation measure?** Internal cohesion/separation of non-noise spatial clusters, alongside cluster coverage, noise, and cluster-size distribution.
- **Will a high Silhouette Score prove that a cluster is a violation hotspot?** No. It will describe geometric separation in the supplied spatial records only.

### Data Analysis Key Findings

- Findings will be completed from the actual eligible Red/Black input after the notebook is run; no results are assumed in advance.

### Insights or Next Steps

- Confirm that the uploaded export exactly matches the production DBSCAN input and selected filters before interpreting the scores.
- Consider any parameter change only after reviewing score, coverage, noise, cluster sizes, and field usefulness together; this notebook will not modify production.